## Crew allocation and EPANET controls
This notebook is meant to test variables and parameters when generating Crew allocation and EPANET controls.
Crew allocation to the pipes (indexes) is generated in random order.
It also imports the functions from crews_and_controls_BPDRR.py 

In [9]:
from pprint import pprint
import random
import pandas as pd
import numpy as np

import crews_and_controls_BPDRR

In [10]:
# Damage scenario selection
ds_sel = 'DS2'

df = pd.read_excel(
    "DS_with_full_description.xlsx",
    sheet_name= ds_sel
)

# Build the reparations dictionary
time_reparation = {}

for _, row in df.iterrows():
    time_reparation[str(row["Pipe ID"])] = {
        "t_r": float(row["fix time (hours)"])
    }

print(f"{len(time_reparation)} repairs loaded.")
print(list(time_reparation.items())[5:10])

106 repairs loaded.
[('6005', {'t_r': 5.724455894848762}), ('1252', {'t_r': 5.724455894848762}), ('2408', {'t_r': 5.724455894848762}), ('3094', {'t_r': 5.724455894848762}), ('3293', {'t_r': 5.724455894848762})]


In [11]:
# load the wdn as INP file WITH the broken pipes
input_inp = 'BBM-EPS_'+ds_sel+'mcg.inp'

# Export the new INP file with the controls
output_inp="BBM-EPS_"+ds_sel+"_restoration.inp"

In [12]:
# Number of crews
n_teams = 3

# Pipe IDs from the reparations dictionary
pipe_ids = list(time_reparation.keys())

# Number of repairs
n_rep = len(pipe_ids)

In [13]:
# Import the travel time matrix from the Excel file
dmatrix_df = pd.read_excel(
    "TravelTime_matrices.xlsx",
    sheet_name= ds_sel,
    index_col=0
)

# Normalize IDs so the repair dictionary and matrix use the same keys
dmatrix_df.index = dmatrix_df.index.map(str)
dmatrix_df.columns = dmatrix_df.columns.map(str)

dmatrix_df.head()

,1951,3414,4988,5251,3404,6005,1252,2408,3094,3293,...,538,5488,5567,5677,5778,5959,6041,854,869,892
1951,0.001335,0.174633,0.059203,0.502553,0.230939,0.574494,0.285973,0.097994,0.227610,0.195529,...,0.248812,0.535616,0.602551,0.510286,0.603958,0.632219,0.577186,0.272680,0.270902,0.270805
3414,0.174633,0.002450,0.159756,0.454119,0.057605,0.526059,0.182835,0.120571,0.053192,0.021110,...,0.135167,0.487181,0.554117,0.461851,0.555523,0.583784,0.528751,0.179511,0.177734,0.157159
4988,0.059203,0.159756,0.006098,0.475991,0.216062,0.547932,0.247441,0.051790,0.212733,0.180652,...,0.216847,0.509054,0.575990,0.483724,0.577396,0.605657,0.550624,0.234147,0.232370,0.238483
5251,0.502553,0.454119,0.475991,0.008997,0.510425,0.091917,0.430311,0.448491,0.507096,0.475014,...,0.399718,0.053039,0.119975,0.027709,0.121381,0.149642,0.094609,0.417018,0.415240,0.421353
3404,0.230939,0.057605,0.216062,0.510425,0.006540,0.582365,0.237347,0.176877,0.069226,0.040278,...,0.189679,0.543487,0.610423,0.518157,0.611829,0.640090,0.585058,0.234023,0.232246,0.211671


In [14]:
indexes = random.sample(pipe_ids, n_rep)
print('Show permutation of reparations')
print(indexes)
print(len(indexes))

Show permutation of reparations
['5140', '2408', '3414', '4721', '3184', '4594', '5488', '3760', '1373', '2307', '5691', '4035', '869', '3875', '2053', '5251', '6041', '3074', '2983', '2730', '4788', '3959', '2881', '3404', '4022', '3216', '538', '1657', '2440', '2622', '6023', '3859', '1477', '4062', '6005', '4622', '3041', '4880', '2910', '2016', '4764', '5567', '404', '2508', '2409', '2113', '2543', '1105', '2816', '4988', '854', '2114', '1402', '5105', '3726', '4882', '4132', '4538', '501', '2001', '762', '892', '4584', '2216', '5871', '4083', '3679', '3564', '2141', '4915', '4177', '4101', '190', '3125', '1807', '1186', '1989', '1724', '3449', '1252', '163', '4507', '2821', '2094', '3122', '1568', '3120', '4038', '4246', '1865', '2726', '2430', '1464', '5775', '3104', '5778', '4409', '2535', '288', '5959', '5677', '1951', '3094', '5042', '4942', '3293']
106


In [15]:
# apply the greedy allocation to the dataset
crews, new_controls = crews_and_controls_BPDRR.allocate_crews(time_reparation, dmatrix_df, indexes, n_teams = n_teams)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
print('')
print('[CONTROLS]')
pprint(new_controls)


organize distances for each crew
Crew 1, from 5140-to-4721 : 0.1709487804878049
Crew 1, from 4721-to-3184 : 0.3690692682926829
Crew 2, from 2408-to-4594 : 0.3396665853658536
Crew 3, from 3414-to-5488 : 0.4871812195121952
Crew 1, from 3184-to-3760 : 0.06703268292682928
Crew 2, from 4594-to-1373 : 0.3345014634146341
Crew 3, from 5488-to-2307 : 0.4934899999999998
Crew 2, from 1373-to-5691 : 0.4606226829268293
Crew 1, from 3760-to-4035 : 0.03821658536585366
Crew 3, from 2307-to-869 : 0.2303729268292682
Crew 2, from 5691-to-3875 : 0.4738863414634146
Crew 1, from 4035-to-2053 : 0.1761926829268293
Crew 3, from 869-to-5251 : 0.41524
Crew 1, from 2053-to-6041 : 0.5558363414634148
Crew 2, from 3875-to-3074 : 0.06279097560975609
Crew 1, from 6041-to-2983 : 0.5906785365853658
Crew 2, from 3074-to-2730 : 0.06867731707317072
Crew 3, from 5251-to-4788 : 0.104679512195122
Crew 2, from 2730-to-3959 : 0.1150419512195122
Crew 1, from 2983-to-2881 : 0.08095878048780489
Crew 2, from 3959-to-3404 : 0.10400

In [16]:
# Apply function to generate INP with controls
controls_inp = crews_and_controls_BPDRR.write_inp_controls(
    input_inp=input_inp,
    output_inp=output_inp,
    crews=crews,
    new_controls=new_controls
)

print(f"Created: {controls_inp}")

Restoration INP successfully created.
Output file : BBM-EPS_DS2_restoration.inp
Controls    : 424
Created: BBM-EPS_DS2_restoration.inp
